# 06 — Information Extraction with `ai_extract`

## Objective

Understand how a Databricks AI Function can pull specific, named fields out of a document's text -- not just a single category label (Notebook 05), but several structured values at once. This notebook extracts fields like `department` and `effective_date` from the 15 documents in Notebook 02, which already carry those exact values as hand-assigned metadata -- so, as in Notebook 05, `ai_extract`'s output can be checked against known ground truth instead of just eyeballed.

## What We Will Learn

- How `ai_extract` turns free text plus a list of field names into a structured record with one value per field
- How this differs from `ai_classify`: one call returns *several* values instead of exactly one label from a fixed set
- How to design an extraction schema -- which fields to ask for, and what it means when a field genuinely isn't present in the text
- How to validate extracted fields against ground truth, and how to interpret a `NULL` extraction result (field absent from the text) versus a wrong one (field present, extracted incorrectly)
- Why the same fixed schema does not fit every document category, and what changes when the schema is tailored per category instead

## Prerequisites

- Completed `02_synthetic_data_generation.ipynb` with matching catalog/schema widget values -- this notebook reads the `documents` Delta table it created
- `ai_extract` is a **preview / entitlement-gated AI Function**, same family as `ai_parse_document` (Notebook 04) and `ai_classify` (Notebook 05) -- it requires Unity Catalog, a supported Databricks Runtime, and availability in your workspace's region. If it isn't enabled, the extraction queries below will fail with a function-not-found or permission error; see **Common Errors / Limitations**
- This notebook has not been executed against a live Databricks workspace. `ai_extract`'s exact return shape (a `STRUCT` with one field per requested label, accessed by dot notation) is documented behavior, but you should confirm it against your workspace's actual output the first time you run Step 3 below, before trusting the field-access code that follows it
- Optional: `05_document_classification_ai_classify.ipynb`, for the classification-vs-ground-truth pattern this notebook extends into per-field validation

## Conceptual Explanation

**What `ai_extract` does.** It is a Databricks-native AI Function, same family as `ai_parse_document` and `ai_classify`, that takes a piece of text and an array of field names, and returns a structured record with one value per field. Where `ai_classify` returns exactly one string from a fixed set of labels, `ai_extract` returns several values at once -- and each one is either a string pulled from the text, or `NULL` if that field genuinely isn't mentioned. It's still an LLM call under the hood, but the *shape* of the answer is fixed by the field list you supply, the same way the *set* of possible answers was fixed by the label array in Notebook 05.

**One call, many fields -- and why that's a different kind of task than classification.** Classification picks among a small number of known, mutually exclusive answers. Extraction has no such constraint: `department` and `effective_date` are independent of each other, a document can be missing one and have the other, and there's no fixed universe of "correct" department names for the model to choose from the way there was a fixed universe of five categories. This makes extraction harder to validate at a glance -- Notebook 05's "did it pick the right one of five options" becomes "did it find the right substring, exactly, for each of several independent fields."

**A field can only be extracted if it's actually in the text.** This sounds obvious, but it's the first thing worth checking, not assuming. Notebook 02's `documents` table keeps `department` and `created_date` as separate metadata *columns* -- the body `content` column itself (a checking-account description, a wire transfer procedure) never states its own department name or date inline. Notebook 05 didn't care, because `ai_classify` only needs topical signal to pick a category. Extraction is different: asking `ai_extract` for `department` against bare `content` would be asking it to find a fact that structurally isn't there, and any "match" against ground truth would be luck, not extraction. So Step 2 below builds each document's text the way a real document actually carries this kind of metadata -- as a short header block (title, category, department, created date) in front of the body -- mirroring exactly the `.txt` file format Notebook 02 already wrote to the volume. That's what makes the ground-truth comparison in Step 4 meaningful.

**Schema design is the real engineering work.** The array of field names you pass to `ai_extract` *is* the schema -- there's no separate schema definition step, no target table DDL to keep in sync. But that field list is doing real work: `'effective_date'` and `'date this policy takes effect'` can pull different substrings out of the same document, the same way label wording changed `ai_classify`'s answers in Notebook 05. And a field that makes sense for one document category -- `'product_name'` for a Product document -- is meaningless for a Compliance policy, which raises a design question this notebook deliberately walks through: one fixed schema for every document, or a schema chosen per category?

**`NULL` is a real, meaningful answer.** If you ask `ai_extract` for a field that the text never mentions, a well-behaved extraction returns `NULL` rather than guessing or hallucinating a value. That makes `NULL` worth treating as a first-class outcome in validation, not an error -- and worth telling apart from the *wrong but non-null* case, where the model found some text and extracted an incorrect value. Step 5 below asks for a field that is deliberately absent from every document's text, specifically so you can see whether `NULL` is what actually comes back in your workspace.

## Example Data

The same 15 documents from `02_synthetic_data_generation.ipynb` used in Notebook 05, with one difference in how this notebook presents them: instead of extracting from the bare `content` column, Step 2 rebuilds each document's text as `"Title: ...\nCategory: ...\nDepartment: ...\nCreated: ...\n\n{content}"` -- the same header-plus-body shape Notebook 02 already wrote to `.txt` files in the volume. `department` and `created_date` remain the ground-truth columns this notebook validates `ai_extract` against.

Additionally, the same **Fraud Escalation Playbook** text reconstructed in Notebook 05 from the Notebook 04 PDF -- used as-is, with no header and no `department` or `created_date` ground truth, which makes it a useful case for watching how `ai_extract` behaves when a requested field is simply not present in the source text at all.

## Implementation

### Step 1 — Point at the same catalog/schema as Notebooks 02 and 05

In [ ]:
dbutils.widgets.text("catalog_name", "main", "Unity Catalog catalog")
dbutils.widgets.text("schema_name", "genai_lab", "Schema")

catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")
documents_table = f"{catalog_name}.{schema_name}.documents"

print(f"Documents table: {documents_table}")

### Step 2 — Load the documents and rebuild the header-plus-body text

`department` and `created_date` are the ground truth we hand-assigned in Notebook 02. They live in their own columns, not inside `content` -- so before extracting anything, we prepend a short metadata header to each document's body, the same shape Notebook 02 wrote to `.txt` files. `ai_extract` will only ever see the combined `document_text` column; the `department` and `created_date` columns are kept aside, untouched, purely as ground truth to check the extraction against in Step 4.

In [ ]:
from pyspark.sql.functions import concat_ws, col, lit

documents_df = spark.table(documents_table)

documents_with_text_df = documents_df.withColumn(
    "document_text",
    concat_ws(
        "",
        lit("Title: "), col("title"), lit("\n"),
        lit("Category: "), col("category"), lit("\n"),
        lit("Department: "), col("department"), lit("\n"),
        lit("Created: "), col("created_date"), lit("\n\n"),
        col("content"),
    ),
)
documents_with_text_df.createOrReplaceTempView("documents_v")

# Sanity check: confirm the header actually landed in document_text before extracting from it
print(documents_with_text_df.select("document_text").first()["document_text"])

### Step 3 — Extract a fixed, two-field schema from every document

`ai_extract(text, fields)` takes the text to extract from and an array of field names, and returns a `STRUCT` with one value per field, accessed by dot notation (`.department`, `.effective_date`). We ask for the same two fields on every document regardless of category -- the simplest possible schema, and the one most directly comparable to ground truth.

**Run this cell first and inspect the raw output before trusting the dot-notation access in later cells** -- confirm your workspace actually returns a `STRUCT` shaped the way this notebook assumes.

In [ ]:
%sql
SELECT
  doc_id,
  title,
  department AS actual_department,
  created_date AS actual_effective_date,
  ai_extract(
    document_text,
    ARRAY('department', 'effective_date')
  ) AS extracted
FROM documents_v
ORDER BY doc_id

### Step 4 — Capture the result in Python and score each field against ground truth

Same query, kept as a DataFrame so we can pull the `department` and `effective_date` fields out of the `extracted` struct individually and compare each one to its ground-truth column. Note the two fields are scored separately -- a document can get `department` right and `effective_date` wrong, or vice versa, in a way that a single classification accuracy number couldn't express.

In [ ]:
extracted_df = spark.sql(
    """
    SELECT
      doc_id,
      title,
      department AS actual_department,
      created_date AS actual_effective_date,
      ai_extract(document_text, ARRAY('department', 'effective_date')) AS extracted
    FROM documents_v
    """
).select(
    "doc_id",
    "title",
    "actual_department",
    "actual_effective_date",
    "extracted.department AS extracted_department",
    "extracted.effective_date AS extracted_effective_date",
).cache()

total = extracted_df.count()
department_correct = extracted_df.filter("actual_department = extracted_department").count()
date_correct = extracted_df.filter("actual_effective_date = extracted_effective_date").count()

print(f"department match:      {department_correct}/{total} ({100 * department_correct / total:.1f}%)")
print(f"effective_date match:  {date_correct}/{total} ({100 * date_correct / total:.1f}%)")

print("\ndepartment mismatches (worth reading, not just counting):")
display(extracted_df.filter("actual_department != extracted_department OR extracted_department IS NULL"))

print("\neffective_date mismatches:")
display(extracted_df.filter("actual_effective_date != extracted_effective_date OR extracted_effective_date IS NULL"))

### Step 5 — Extract a field that genuinely is not in the text

Even with the metadata header from Step 2, none of these 15 documents state a regulatory filing number anywhere. Ask for that field and see whether `ai_extract` returns `NULL` (the correct, honest answer) rather than inventing one.

In [ ]:
absent_field_df = spark.sql(
    """
    SELECT
      doc_id,
      title,
      ai_extract(document_text, ARRAY('regulatory_filing_number')).regulatory_filing_number AS extracted_filing_number
    FROM documents_v
    """
).cache()

null_count = absent_field_df.filter("extracted_filing_number IS NULL").count()
total = absent_field_df.count()
print(f"NULL (correctly absent): {null_count}/{total}")

print("\nAny non-null value here is worth reading closely -- it means the model found or invented something:")
display(absent_field_df.filter("extracted_filing_number IS NOT NULL"))

### Step 6 — Tailor the schema per category instead of using one fixed schema

`department` and `effective_date` make sense for every document. `product_name` only makes sense for a Product document; `policy_type` only makes sense for a Compliance document. This step asks each category's documents for a field specific to that category, alongside the two universal fields -- a direct look at the one-fixed-schema-for-everything (Step 3) versus schema-per-category trade-off raised in the Conceptual Explanation. Unlike Steps 3-4, these category-specific fields have no ground-truth column to score against -- read the values, don't just count them.

In [ ]:
category_fields = {
    "Product": "product_name",
    "Operations": "applicable_process",
    "Compliance": "policy_type",
    "Customer Service": "applicable_process",
    "Technical": "api_resource",
}

per_category_results = []
for category, extra_field in category_fields.items():
    rows = spark.sql(
        f"""
        SELECT
          doc_id,
          title,
          category,
          ai_extract(document_text, ARRAY('department', 'effective_date', '{extra_field}')) AS extracted
        FROM documents_v
        WHERE category = '{category}'
        """
    ).select(
        "doc_id",
        "title",
        "category",
        "extracted.department AS department",
        "extracted.effective_date AS effective_date",
        f"extracted.{extra_field} AS {extra_field}",
    )
    per_category_results.append((category, extra_field, rows))

for category, extra_field, rows in per_category_results:
    print(f"\n{category} -- extra field: {extra_field}")
    display(rows)

### Step 7 — Extract from the unlabeled Fraud Escalation Playbook

This document has no metadata header and no `department` or `created_date` ground truth at all -- a genuine test of whether `ai_extract` returns `NULL` for a field that is truly absent from the source text, on a document this notebook has no prior expectation for.

In [ ]:
fraud_escalation_text = (
    "Fraud Escalation Playbook\n\n"
    "This playbook defines how Aurora Trust Bank staff escalate suspected fraud cases. "
    "Any transaction flagged by fraud monitoring must be reviewed within one business hour. "
    "Confirmed fraud cases are escalated to the Fraud Operations team, who freeze the affected "
    "account and notify the customer through the contact center. "
    "Escalation Steps: (1) Analyst reviews the flagged transaction. (2) Analyst confirms or "
    "dismisses the fraud indicator. (3) Confirmed cases are routed to Fraud Operations. "
    "(4) Fraud Operations freezes the account and opens a case file. "
    "Escalation Tiers: Tier 1 handles amounts under 1,000 units; Tier 2 handles 1,000-10,000 "
    "units and requires a supervisor sign-off; Tier 3 handles amounts above 10,000 units and "
    "requires notifying the Compliance department."
)

fraud_df = spark.createDataFrame([("fraud_escalation_playbook", fraud_escalation_text)], ["title", "document_text"])
fraud_df.createOrReplaceTempView("fraud_doc_v")

display(
    spark.sql(
        """
        SELECT
          title,
          ai_extract(
            document_text,
            ARRAY('department', 'effective_date', 'escalation_tier_count')
          ) AS extracted
        FROM fraud_doc_v
        """
    )
)

## Inspect the Output

- What was the `department` and `effective_date` match rate in Step 4? Are the two fields equally reliable, or does one extract more accurately than the other?
- For any mismatch in Step 4, did `ai_extract` return a *plausible but wrong* value, or `NULL`? Those are different failure modes worth telling apart, not a single "wrong" bucket.
- Did Step 5 come back `NULL` for every document, as expected? If your workspace returned a non-null value for `regulatory_filing_number`, read it closely -- that's the extraction equivalent of the "invalid prediction" check in Notebook 05, and worth taking seriously.
- In Step 6, did the category-specific field (`product_name`, `policy_type`, etc.) extract cleanly from the body content, without a dedicated header line the way `department` had one? Does that make you trust it less than the Step 4 fields?
- What came back for the Fraud Escalation Playbook in Step 7? The text mentions "Fraud Operations team" and "Compliance department" in passing but states neither as a formal `department` field -- does `ai_extract` pick one, combine them, or return `NULL`?

## Experimentation Section

1. Re-run Step 3 with a more descriptive field name -- `'department'` vs `'the internal department or team responsible for this document'` -- the same label-wording experiment Notebook 05 ran on `ai_classify`. Does wording change which mismatches from Step 4 disappear?
2. Re-run Step 3 against the bare `content` column instead of `document_text` (no header) and compare the match rate to Step 4's. This isolates exactly how much of Step 4's accuracy came from the header you added versus the model inferring department from body content alone.
3. Add a field to Step 6 that has no good answer in *any* category (e.g. `'customer_complaint_id'`) and confirm it comes back `NULL` across every category, not just the ones it's obviously irrelevant for.
4. Pick one document and request ten fields at once, several of which are clearly irrelevant to that document's content -- does extraction quality degrade as the field list grows, or does `ai_extract` handle irrelevant fields gracefully?
5. Combine this notebook with Notebook 05: first `ai_classify` a document, then use the predicted category to choose which per-category extra field to request from `ai_extract`, chaining the two AI Functions in one query.

## Common Errors / Limitations

- **`UNRESOLVED_ROUTINE` / function not found** -- `ai_extract` isn't enabled for your workspace, region, or SQL warehouse/cluster type. Same remedy as `ai_parse_document` (Notebook 04) and `ai_classify` (Notebook 05): check release notes or ask a workspace admin.
- **Permission or entitlement errors** -- the function exists but your workspace isn't entitled to call it. Same remedy -- check with an admin.
- **Return shape may differ from what this notebook assumes** -- this notebook has not been executed against a live workspace, and treats `ai_extract`'s output as a `STRUCT` accessed by dot notation. Confirm this against your actual Step 3 output before relying on the field-access code in Steps 4, 6, and 7; if your workspace returns a `MAP` instead, the access syntax changes (e.g. `extracted['department']`) but the concepts below are unaffected.
- **A field must actually be present in the text to be extractable** -- Step 2 exists specifically because `department` and `created_date` are not inline in Notebook 02's `content` column. If you ever extract from a different corpus and see suspiciously poor accuracy, check whether the field is genuinely stated in the text before assuming `ai_extract` failed.
- **`NULL` is not always distinguishable from "extraction failed"** -- a `NULL` result should mean "this field isn't in the text," but without ground truth (as in Steps 6-7) you can't be fully certain the model didn't simply miss something that was there. This is exactly why Steps 3-4 use fields with known ground truth first.
- **Extraction is not free** -- like `ai_parse_document` and `ai_classify`, each `ai_extract` call is a managed AI service invocation, and asking for more fields per call is not necessarily cheaper than several smaller calls. Don't loop it row-by-row over a large table without first confirming cost/throughput expectations; the `SELECT ... FROM documents_v` pattern above already applies it set-at-a-time.
- **Accuracy on 15 documents is not a benchmark** -- same caveat as Notebook 05: this dataset is sized for inspectability, not statistical significance.
- **A fixed schema silently under-serves categories it wasn't designed for** -- Step 3's two-field schema works for every document only because `department` and `effective_date` are universal in this synthetic dataset (once given a header to live in). Nothing about `ai_extract` itself enforces that a schema you design for one document type will make sense for another; that judgment call is on you, same as it was for `ai_classify`'s labels.

## Summary

You extracted structured fields from 15 known documents with `ai_extract` and measured each field's accuracy against ground truth separately -- after first making sure the fields you asked for were actually present in the text, by rebuilding a metadata header the way Notebook 02's `.txt` files already carried it. You watched a genuinely absent field return `NULL` rather than a hallucinated value, compared one fixed schema applied to every document against a schema tailored per category, and extracted from an unlabeled, ambiguous document with no ground truth at all. The core takeaway, building directly on Notebook 05: where `ai_classify` moves the engineering work into *label wording*, `ai_extract` moves it into *schema design* -- deciding which fields to ask for, confirming they're actually recoverable from the text, and treating `NULL` as a meaningful answer rather than a missing one.

## Suggested Exercises

- For each of the five categories, write down one field beyond `department` and `effective_date` that you'd actually want extracted in a real system -- then check whether Step 6's `category_fields` dict already covers it, and add it if not.
- Pick one mismatch from Step 4 and decide, in your own judgment, whether the *document* text is genuinely ambiguous about that field, or the *model* got it wrong. Justify your answer using only `document_text`, the same exercise Notebook 05 asked for classification.
- When you're ready, move on to **`07_document_chunking_basics.ipynb`**, which stops asking "what does this document say about field X" and starts asking how to break a document into pieces small enough to retrieve and reason over at all.